In [ ]:
import imaplib
import email
from email.header import decode_header
from email.utils import parseaddr
import pandas as pd
import re
import os

# 1. Configurazione Account
# L'approccio migliore per gli script in produzione è usare variabili d'ambiente
# Se usi un file .env, puoi decommentare le due righe seguenti:
# from dotenv import load_dotenv
# load_dotenv()

ACCOUNTS = [
    {
        "email": "TUA_MAIL@gmail.com",
        "password": os.getenv("GMAIL_PWD_00", "PWD_1"), # Inserisci nel .env o usa default
        "server": "imap.gmail.com"
    },
    {
        "email": "TUA_MAIL@gmail.com",
        "password": os.getenv("GMAIL_PWD_01", "PWD_2"),
        "server": "imap.gmail.com"
    },
]

def clean_company_name(sender_string):
    """Estrae il nome dell'azienda pulito dal campo mittente dell'email."""
    if not sender_string:
        return "Sconosciuta"
        
    # 1. Decodifica header (gestione stringhe =?utf-8?q?...)
    if "=?" in sender_string:
        try:
            decoded_parts = decode_header(sender_string)
            sender_string = "".join([
                part[0].decode(part[1] or 'utf-8', errors='ignore') if isinstance(part[0], bytes) else part[0] 
                for part in decoded_parts
            ])
        except Exception as e:
            print(f"Errore di decodifica header per {sender_string}: {e}")

    # 2. Estrazione sicura nativa con parseaddr (es. restituisce ('Nome Azienda', 'email@dominio.com'))
    name, email_addr = parseaddr(sender_string.replace('"', '').replace("'", ""))
    company_name = name.strip()
    
    # 3. Fallback sul dominio se il nome è vuoto o un sistema ATS generico
    generic_names = {"no-reply", "noreply", "hr", "recruiting", "careers", "info", "system", "hr system", "talent acquisition"}
    
    if not company_name or company_name.lower() in generic_names:
        if "@" in email_addr:
            domain = email_addr.split('@')[1].split('.')[0].lower()
            if domain in ["successfactors", "myworkday", "global", "bendingspoons"]:
                return domain.capitalize()
            return domain.capitalize()
        return sender_string.strip()
        
    # Pulizia finale dei suffissi
    company_name = re.sub(r'(?i)\s*(careers|recruiting|hr|human resources|talent acquisition).*', '', company_name)
    
    return company_name.strip()

def get_email_body(msg):
    """Estrae il testo pulito dal corpo dell'email"""
    if msg.is_multipart():
        for part in msg.walk():
            content_type = part.get_content_type()
            content_disposition = str(part.get("Content-Disposition"))
            if content_type == "text/plain" and "attachment" not in content_disposition:
                try:
                    return part.get_payload(decode=True).decode(part.get_content_charset() or 'utf-8', errors='ignore')
                except Exception:
                    continue
    else:
        try:
            return msg.get_payload(decode=True).decode(msg.get_content_charset() or 'utf-8', errors='ignore')
        except Exception:
            pass
    return ""

def classify_email(subject, body, sender):
    sender_lower = str(sender).lower()
    subject_lower = str(subject).lower()
    text_to_analyze = f"{subject_lower} {body.lower()}"
    
    # 0. Filtri Anti-Spam / Personali e Newsletter
    if "github" in sender_lower or "unobravo" in sender_lower:
        return "Ignorato"
    if "soundcloud" in sender_lower or "linkedin" in sender_lower or "newsletter" in sender_lower:
        return "Ignorato (Newsletter/Spam)"
        
    # 1. ESITI NEGATIVI (Priorità massima)
    rejection_keywords = ["purtroppo", "non procederemo", "non selezionato", "unfortunately", "not moving forward"]
    if any(kw in text_to_analyze for kw in rejection_keywords):
        return "Esito Negativo"

    # 2. CONTATTI / OPPORTUNITÀ
    opportunity_keywords = ["opportunità professionale", "opportunita professionale", "contatto", "opportunity", "new role", "job opportunity"]
    if any(kw in subject_lower for kw in opportunity_keywords):
        return "Contatto / Nuova Opportunità"

    # 3. CANDIDATURE RICEVUTE (Controllo rapido su Oggetto)
    confirmation_subjects = ["application received", "thanks for your application", "thank you for applying", "candidatura ricevuta", "conferma"]
    if any(kw in subject_lower for kw in confirmation_subjects):
        return "Candidatura Ricevuta"
        
    # 4. COLLOQUI REALI (Controllo robusto)
    interview_strong_keywords = [
        "invitation to interview", "invite you to", "schedule an interview", 
        "fissare un colloquio", "interview reminder", "calendly.com", "teams.microsoft", 
        "interview you", "phone interview", "interview availability", "colloquio amazon"
    ]
    if any(kw in text_to_analyze for kw in interview_strong_keywords):
        return "Colloquio / Riunione Fissata"

    # 5. CANDIDATURE RICEVUTE (Controllo di riserva nel corpo del testo)
    confirmation_general = ["candidatura", "your application", "job application", "confirming"]
    if any(kw in text_to_analyze for kw in confirmation_general):
        return "Candidatura Ricevuta"
        
    if "esito" in subject_lower:
        return "Esito Ricevuto (Da Leggere)"
        
    return "Da Verificare / Altro"

def scan_account(account_config):
    """Si connette a un singolo account e scarica le email rilevanti"""
    print(f"\nConnessione in corso a: {account_config['email']}...")
    
    try:
        mail = imaplib.IMAP4_SSL(account_config['server'])
        mail.login(account_config['email'], account_config['password'])
        mail.select("inbox")
        
        search_query = '(OR SUBJECT "application" (OR SUBJECT "candidatura" (OR SUBJECT "colloquio" (OR SUBJECT "interview" SUBJECT "esito"))))'
        status, messages = mail.search(None, search_query)
        email_ids = messages[0].split()
        
        account_data = []
        print(f"Trovate {len(email_ids)} email da analizzare su questo account.")

        for e_id in email_ids:
            res, msg_data = mail.fetch(e_id, "(RFC822)")
            for response in msg_data:
                if isinstance(response, tuple):
                    msg = email.message_from_bytes(response[1])
                    
                    subject, encoding = decode_header(msg["Subject"])[0]
                    if isinstance(subject, bytes):
                        subject = subject.decode(encoding if encoding else "utf-8", errors='ignore')
                    
                    sender = msg.get("From")
                    date = msg.get("Date")
                    body = get_email_body(msg)
                    
                    azienda_pulita = clean_company_name(sender)
                    stato = classify_email(subject, body, sender)
                    
                    if stato not in ["Ignorato", "Ignorato (Newsletter/Spam)"]:
                        account_data.append({
                            "Account Destinatario": account_config['email'],
                            "Data": date,
                            "Azienda": azienda_pulita,
                            "Mittente Originale": sender,
                            "Oggetto": subject,
                            "Stato Classificato": stato
                        })
                        
        mail.logout()
        return account_data

    except Exception as e:
        print(f"Errore durante l'accesso a {account_config['email']}: {e}")
        return []

if __name__ == "__main__":
    tutti_i_dati = []
    
    for account in ACCOUNTS:
        tutti_i_dati.extend(scan_account(account))
        
    df = pd.DataFrame(tutti_i_dati)
    
    if not df.empty:
        # A. Parsing date ottimizzato
        df['Data_Parsed'] = pd.to_datetime(df['Data'], errors='coerce', utc=True, format='mixed')
        
        # B. Vettorizzazione dell'estrazione del dominio e del giorno con assign()
        df = df.assign(
            Giorno=df['Data_Parsed'].dt.date,
            Azienda_Base=df['Azienda'].str.lower().str.split('@').str[-1].str.split('.').str[0]
        )
        
        # C. Gestione duplicati sui colloqui
        is_colloquio = df['Stato Classificato'] == 'Colloquio / Riunione Fissata'
        duplicates_mask = df[is_colloquio].duplicated(subset=['Giorno', 'Azienda_Base'], keep='first')
        
        # D. Aggiornamento stato
        df.loc[df[is_colloquio][duplicates_mask].index, 'Stato Classificato'] = 'Colloquio (Update/Reminder stesso giorno)'
        
        # E. Drop pulito delle colonne di supporto temporanee
        df = df.drop(columns=['Data_Parsed', 'Giorno', 'Azienda_Base'])
        
        print("\n--- Riepilogo Classificazioni ---")
        print(df["Stato Classificato"].value_counts())
        
        df.to_csv("report_candidature_definitivo.csv", index=False, encoding="utf-8-sig")
        print("\nExport completato: report_candidature_definitivo.csv")
    else:
        print("\nNessun dato trovato o elaborato.")